# Sentence Embedding

Chuyển một câu (chuỗi ký tự) thành ma trận embedding đầu vào cho encoder/decoder: tokenize theo từng ký tự → tra bảng embedding → cộng positional encoding → dropout.

Viết hoàn toàn bằng `numpy`, không dùng PyTorch. Bản gốc dùng `nn.Module`, `nn.Embedding`, `nn.Dropout`, `torch.tensor`/`torch.stack` và `.to(get_device())` — ở đây thay bằng các lớp numpy tương ứng (không cần quản lý device vì numpy chỉ chạy CPU).

In [1]:
import numpy as np

Các khối cơ bản: `Parameter` bọc dữ liệu trọng số; `Embedding` là bảng tra cứu vector cho từng chỉ số token (thay cho `nn.Embedding`); `Dropout` tự cài bằng numpy (thay cho `nn.Dropout`).

In [2]:
class Parameter:
    def __init__(self, data):
        self.data = data
        self.grad = None


class Embedding:
    def __init__(self, num_embeddings, embedding_dim):
        self.weight = Parameter(np.random.randn(num_embeddings, embedding_dim))

    def __call__(self, indices):
        return self.weight.data[indices]


class Dropout:
    def __init__(self, p):
        self.p = p
        self.training = True

    def __call__(self, x):
        if not self.training or self.p == 0:
            return x
        mask = (np.random.rand(*x.shape) > self.p).astype(x.dtype)
        return x * mask / (1 - self.p)

`PositionalEncoding` — giống các notebook trước (xem notebook Positional Encoding), dùng để mã hoá vị trí token.

In [3]:
class PositionalEncoding:
    def __init__(self, d_model, max_sequence_length):
        self.d_model = d_model
        self.max_sequence_length = max_sequence_length

    def forward(self):
        even_i = np.arange(0, self.d_model, 2).astype(np.float32)
        even_dominator = np.power(10000, even_i / self.d_model)

        odd_i = np.arange(1, self.d_model, 2).astype(np.float32)
        odd_dominator = np.power(10000, (odd_i - 1) / self.d_model)

        denominator = even_dominator

        position = np.arange(self.max_sequence_length, dtype=np.float32).reshape(self.max_sequence_length, 1)

        even_PE = np.sin(position / denominator)
        odd_PE = np.cos(position / denominator)

        stacked = np.stack((even_PE, odd_PE), axis=2)
        PE = stacked.reshape(stacked.shape[0], -1)

        return PE

`SentenceEmbedding`: tokenize từng ký tự trong câu thành chỉ số (thêm `START_TOKEN`/`END_TOKEN`, đệm `PADDING_TOKEN` cho đủ `max_sequence_length`) → tra embedding → cộng positional encoding → dropout. `torch.tensor`/`torch.stack`/`.to(get_device())` → `np.array`/`np.stack`.

In [4]:
class SentenceEmbedding:
    "For a given sentence, create an embedding"
    def __init__(self, max_sequence_length, d_model, language_to_index, START_TOKEN, END_TOKEN, PADDING_TOKEN):
        self.vocab_size = len(language_to_index)
        self.max_sequence_length = max_sequence_length
        self.embedding = Embedding(self.vocab_size, d_model)
        self.language_to_index = language_to_index
        self.position_encoder = PositionalEncoding(d_model, max_sequence_length)
        self.dropout = Dropout(p=0.1)
        self.START_TOKEN = START_TOKEN
        self.END_TOKEN = END_TOKEN
        self.PADDING_TOKEN = PADDING_TOKEN

    def batch_tokenize(self, batch, start_token, end_token):

        def tokenize(sentence, start_token, end_token):
            sentence_word_indicies = [self.language_to_index[token] for token in list(sentence)]
            if start_token:
                sentence_word_indicies.insert(0, self.language_to_index[self.START_TOKEN])
            if end_token:
                sentence_word_indicies.append(self.language_to_index[self.END_TOKEN])
            for _ in range(len(sentence_word_indicies), self.max_sequence_length):
                sentence_word_indicies.append(self.language_to_index[self.PADDING_TOKEN])
            return np.array(sentence_word_indicies)

        tokenized = []
        for sentence_num in range(len(batch)):
            tokenized.append(tokenize(batch[sentence_num], start_token, end_token))
        tokenized = np.stack(tokenized)
        return tokenized

    def forward(self, x, start_token, end_token): # sentence
        x = self.batch_tokenize(x, start_token, end_token)
        x = self.embedding(x)
        pos = self.position_encoder.forward()
        x = self.dropout(x + pos)
        return x

### Input

Vocabulary theo ký tự (character-level). `max_sequence_length=10` để `<START>`/`<END>`/`<PAD>` vừa đủ cho 2 câu ví dụ.

In [5]:
START_TOKEN = '<START>'
END_TOKEN = '<END>'
PADDING_TOKEN = '<PAD>'

vocabulary = [START_TOKEN, END_TOKEN, PADDING_TOKEN] + list("abcdefghijklmnopqrstuvwxyz ")
language_to_index = {token: index for index, token in enumerate(vocabulary)}

d_model = 512
max_sequence_length = 10
batch = ["hello", "hi there"]

model = SentenceEmbedding(max_sequence_length, d_model, language_to_index, START_TOKEN, END_TOKEN, PADDING_TOKEN)
out = model.forward(batch, start_token=True, end_token=True)
out.shape, type(out)

((2, 10, 512), <class 'numpy.ndarray'>)

Kiểm tra token hoá: câu `"hello"` (5 ký tự) được bọc `<START>`/`<END>` rồi đệm `<PAD>` cho đủ 10; câu `"hi there"` (8 ký tự, có khoảng trắng) vừa đủ 10, không cần đệm.

In [6]:
model.batch_tokenize(batch, start_token=True, end_token=True)

array([[ 0, 10,  7, 14, 14, 17,  1,  2,  2,  2],
       [ 0, 10, 11, 29, 22, 10,  7, 20,  7,  1]])